# Maintainers Copilot — Classical TF-IDF + Logistic Regression Baseline

This notebook is intentionally separate from the DistilBERT notebook. It assumes the corrected `train.jsonl`, `val.jsonl`, and `test.jsonl` files already exist, usually in Google Drive at `MyDrive/maintainers-copilot/data/`.

No GPU is needed.

## 1. Install lightweight dependencies

In [ ]:
!pip -q install scikit-learn

## 2. Mount Drive and choose file paths

If your files are not in Drive, upload the `data/` folder into Colab and change `DATA_DIR` to `Path('data')`.

In [ ]:
from __future__ import annotations

import hashlib
import json
from dataclasses import asdict, dataclass
from datetime import UTC, datetime
from pathlib import Path
from time import perf_counter
from typing import Any

from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/maintainers-copilot')
DATA_DIR = DRIVE_ROOT / 'data'
RUN_NAME = 'classical-tfidf-logreg'
RUN_DIR = DRIVE_ROOT / 'artifacts' / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

for required in ['train.jsonl', 'val.jsonl', 'test.jsonl']:
    path = DATA_DIR / required
    if not path.exists():
        raise FileNotFoundError(f'Missing {path}. Copy the corrected dataset files to Drive first.')

print('Using data from:', DATA_DIR)
print('Writing evidence to:', RUN_DIR)

## 3. Shared constants and helpers

In [ ]:
TARGET_LABELS = ('bug', 'feature', 'docs', 'question')
LABEL_TO_ID = {
    'bug': 0,
    'feature': 1,
    'docs': 2,
    'question': 3,
}


@dataclass(frozen=True, slots=True)
class DatasetFingerprint:
    path: str
    sha256: str
    examples: int


@dataclass(frozen=True, slots=True)
class ClassicalBaselineConfig:
    run_name: str = RUN_NAME
    model_family: str = 'tfidf_logistic_regression'
    analyzer: str = 'word'
    ngram_min: int = 1
    ngram_max: int = 2
    max_features: int = 50_000
    min_df: int = 2
    sublinear_tf: bool = True
    lowercase: bool = True
    strip_accents: str = 'unicode'
    logistic_regression_max_iter: int = 1_000
    logistic_regression_c: float = 4.0
    class_weight: str = 'balanced'
    random_state: int = 42


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    with path.open(encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            record = json.loads(line)
            if not isinstance(record, dict):
                raise ValueError(f'Expected an object on line {line_number} of {path}.')
            records.append(record)
    return records


def compose_issue_text(title: str | None, body: str | None) -> str:
    return f'{(title or '').strip()}\n\n{(body or '').strip()}'.strip()


def read_issue_examples(path: Path) -> tuple[list[str], list[str]]:
    records = read_jsonl(path)
    texts = [compose_issue_text(record.get('title'), record.get('body')) for record in records]
    targets = [record['target'] for record in records]
    return texts, targets


def fingerprint_jsonl(path: Path) -> DatasetFingerprint:
    digest = hashlib.sha256()
    examples = 0
    with path.open('rb') as handle:
        for line in handle:
            if line.strip():
                examples += 1
            digest.update(line)
    if examples == 0:
        raise ValueError(f'Dataset split is empty: {path}')
    return DatasetFingerprint(path=str(path), sha256=digest.hexdigest(), examples=examples)

## 4. Train and evaluate the classical baseline

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline

config = ClassicalBaselineConfig()

train_texts, train_targets = read_issue_examples(DATA_DIR / 'train.jsonl')
val_texts, val_targets = read_issue_examples(DATA_DIR / 'val.jsonl')
test_texts, test_targets = read_issue_examples(DATA_DIR / 'test.jsonl')

pipeline = Pipeline(steps=[
    ('tfidf', TfidfVectorizer(
        analyzer=config.analyzer,
        ngram_range=(config.ngram_min, config.ngram_max),
        max_features=config.max_features,
        min_df=config.min_df,
        sublinear_tf=config.sublinear_tf,
        lowercase=config.lowercase,
        strip_accents=config.strip_accents,
    )),
    ('classifier', LogisticRegression(
        max_iter=config.logistic_regression_max_iter,
        C=config.logistic_regression_c,
        class_weight=config.class_weight,
        random_state=config.random_state,
    )),
])

fit_started = perf_counter()
pipeline.fit(train_texts, train_targets)
fit_seconds = perf_counter() - fit_started


def evaluate_split(split_name: str, texts: list[str], actual: list[str]) -> tuple[dict[str, Any], dict[str, Any]]:
    started = perf_counter()
    predicted = list(pipeline.predict(texts))
    predict_seconds = perf_counter() - started
    report = classification_report(
        actual,
        predicted,
        labels=list(TARGET_LABELS),
        output_dict=True,
        zero_division=0,
    )
    metrics = {
        'split': split_name,
        'examples': len(actual),
        'accuracy': float(accuracy_score(actual, predicted)),
        'macro_f1': float(f1_score(actual, predicted, labels=list(TARGET_LABELS), average='macro')),
        'weighted_f1': float(f1_score(actual, predicted, labels=list(TARGET_LABELS), average='weighted')),
        'per_class_f1': {label: float(report[label]['f1-score']) for label in TARGET_LABELS},
        'confusion_matrix_labels': list(TARGET_LABELS),
        'confusion_matrix': confusion_matrix(actual, predicted, labels=list(TARGET_LABELS)).tolist(),
        'predict_seconds': predict_seconds,
        'examples_per_second': len(actual) / predict_seconds if predict_seconds else None,
    }
    return metrics, report

val_metrics, val_report = evaluate_split('validation', val_texts, val_targets)
test_metrics, test_report = evaluate_split('test', test_texts, test_targets)

metrics = {
    'validation': val_metrics,
    'test': test_metrics,
    'timing': {
        'fit_seconds': fit_seconds,
        'validation_predict_seconds': val_metrics['predict_seconds'],
        'test_predict_seconds': test_metrics['predict_seconds'],
    },
    'model_size': {
        'vocabulary_size': len(pipeline.named_steps['tfidf'].vocabulary_),
        'labels': list(TARGET_LABELS),
    },
}
reports = {'validation': val_report, 'test': test_report}
metrics

## 5. Save small evidence files to Drive

Bring these three files back to the repo after the run.

In [ ]:
manifest = {
    'created_at': datetime.now(UTC).isoformat(),
    'experiment': 'classical_baseline',
    'config': asdict(config),
    'labels': LABEL_TO_ID,
    'dataset': {
        'train': asdict(fingerprint_jsonl(DATA_DIR / 'train.jsonl')),
        'val': asdict(fingerprint_jsonl(DATA_DIR / 'val.jsonl')),
        'test': asdict(fingerprint_jsonl(DATA_DIR / 'test.jsonl')),
    },
    'artifact_policy': 'commit small metrics/manifests only; do not commit binary model artifacts',
}

(RUN_DIR / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, sort_keys=True) + '\n', encoding='utf-8')
(RUN_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2, sort_keys=True) + '\n', encoding='utf-8')
(RUN_DIR / 'classification_report.json').write_text(json.dumps(reports, indent=2, sort_keys=True) + '\n', encoding='utf-8')

for path in sorted(RUN_DIR.glob('*.json')):
    print(path)

## 6. Quick result view

In [ ]:
print('Validation accuracy:', metrics['validation']['accuracy'])
print('Validation macro-F1:', metrics['validation']['macro_f1'])
print('Test accuracy:', metrics['test']['accuracy'])
print('Test macro-F1:', metrics['test']['macro_f1'])
print('Per-class test F1:', metrics['test']['per_class_f1'])